# Keys and duplicates

"What identifies a row?" is the first structural question about a table,
and the profiler answers it with measured statements: uniqueness findings
backed by exact full-column counts, and duplicate findings backed by
exact comparisons. Both are `measured`-basis, `full`-scope,
`direct_comparison` evidence — the strongest class in the ledger — unless
the comparison rule itself says otherwise.

## Single-column keys

A column is a key candidate when it is null-free and its distinct count
equals the row count — both computed exactly over the full column, at any
cardinality. Among candidates the profiler ranks by a simple score:
0.4 × completeness + 0.4 × type affinity + 0.2 × name evidence. Integer
and unsigned dtypes score full type affinity, strings 0.8, temporal
columns 0.4, floats 0.2. A name containing a token like `id`, `key`,
`code`, `no`, `num`, `number`, `uuid`, `guid`, `sku`, or `ref` scores the
name component. The top candidate becomes the headline's key candidate;
the score is folklore, not theory — it encodes the observation that
integer columns named `*id` are keys far more often than float columns
named `quantity`.

## Composite keys

When no single column is a key, pairs are screened: null-free columns with
at least two distinct values are hashed over a 2,048-row stratified
sample, and pairs that look unique there — up to 16 candidates (24 on
`deep`), ranked by combined key score — are verified by exact canonical
counting over the full table. Deep budgets continue to triples over the
12 most promising columns not already in a verified pair. A composite-key
finding states the pair that jointly identifies rows, with the exact
verification behind it.

The screen is cheap and the verification is exact; a pair that passes both
is a key on the whole table, not a sample claim.

## Dirty keys

A dirty key is a column that *almost* identifies rows: near-unique, with a
small number of repeated values. The finding carries the duplicates count
and example values, because the repeated values are the payload — a join
on a dirty key silently duplicates rows, and "which values repeat" is the
question the reader asks next. The uniqueness statement and the
identifier-role statement about the same column are deliberately separate:
one is measured, one is inferred, and consolidation does not merge them.

## Row duplicates

Duplicate rows are detected exactly: a full-row comparison, not a hash
match alone — hashes propose, canonical bytes confirm, so a hash
collision can never invent a duplicate. The finding carries the number of
duplicate groups, redundant rows, the largest group, and example rows; the
specification states the comparison explicitly (exact equality, missing
values comparing equal), so the claim is falsifiable by re-running the
comparison.

A unique, null-free column also gives the duplicate pass a shortcut: if some column has already been proven
unique and null-free, duplicate rows are logically impossible, and the
pass reports that instead of scanning — the coverage line says so. The
deduction is sound and the report says which path it took.

In [ ]:
import random

import polars as pl

import onceover

rng = random.Random(5)
n = 3_000
external_ref = [f"R{i:05}" for i in range(n)]
external_ref[-1] = external_ref[0]     # two duplicated values
external_ref[-2] = external_ref[1]
region_id = [rng.randrange(3) for _ in range(n)]
region_name = [["emea", "amer", "apac"][r] for r in region_id]
rows = {
    "order_id": list(range(n)),
    "external_ref": external_ref,
    "region_id": region_id,
    "region_name": region_name,
}
rows.update({f"noise_{i}": [rng.random() for _ in range(n)] for i in range(3)})
df = pl.DataFrame(rows)
df = pl.concat([df, df[100:103]])      # three duplicated rows

report = onceover.profile(df)
print(report.overview["headline"])
for f in report.findings:
    if f["category"] in {"uniqueness", "duplicates"}:
        print(f"[{f['interestingness']:.2f}] {f['claim']}")
        for e in f["evidence"]:
            m = e["metrics"]
            if "examples" in m:
                print("   examples:", m["examples"][:3])

Reading the output: there is no key candidate in the headline, and that
is the first lesson. Appending three duplicated rows dirtied `order_id`,
so no single column identifies a row any more. What the uniqueness pass
reports instead is the fingerprint of those duplicates: every column,
including the random `noise_*` columns, is "nearly unique with 3 repeated
non-null values", and for `order_id` the repeated values are `100`, `101`,
`102` — the same three rows each time. The `duplicates` finding states
that directly: 3 groups, largest 2, with the (row, row) pairs.
`external_ref` carries its own two planted duplications on top of the
appended rows, hence 5 repeated values.

These findings describe related properties, but each answers a different question. A dirty key, an identifier
role, and a duplicate row are three claims about the same evidence, and
the report keeps them separate: the column statements are measured
full-column counts, the duplicate statement is a measured full-row
comparison, and the role is inferred from the counts. The usage chapters
show how these findings drive decisions ([](../usage/quickstart.ipynb)
and the [vendor-extract case study](../case_studies/vendor_extract.ipynb)).